# CV Search Basic Grid Search와 Random Search를 같은 개발 데이터, 4-fold, AP와 후보 12개 조건에서 비교했습니다. 평균 AP뿐 아니라 fold 표준편차와 계산 예산을 함께 확인하고, 사전 선택 규칙으로 후보를 고정한 뒤 sealed Test를 한 번 평가했습니다. 기본 실습 완료. 강의 문제 원문은 제외

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

RANDOM_STATE = 42

# [1] 실제 AI 서비스 로그 대신 재현 가능한 합성 분류 데이터를 만듭니다.
# y=1은 사람이 검토해야 하는 답변이며 전체의 약 12%입니다.
X, y = make_classification(
    n_samples=1600,
    n_features=14,
    n_informative=8,
    n_redundant=2,
    weights=[0.88, 0.12],
    class_sep=0.9,
    random_state=RANDOM_STATE,
)

# [2] 탐색용 개발 데이터와 마지막 평가용 test를 먼저 분리합니다.
# stratify=y는 두 집합의 needs_review=1 비율이 크게 달라지는 것을 줄입니다.
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

# [3] Grid와 Random의 모든 후보가 정확히 같은 네 fold를 사용합니다.
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# [4] 탐색 결과를 보기 전에 후보 선택 규칙을 고정합니다.
# 0.01은 최고 점수의 1%가 아니라 AP의 절대 차이 0.01입니다.
MEAN_TOLERANCE = 0.01
SELECTION_ORDER = [
    "best_std_AP",
    "n_estimators",
    "max_depth",
    "method",
]

# [5] 두 탐색 방식이 공유할 기본 Random Forest입니다.
# class_weight="balanced"는 드문 needs_review=1에 더 큰 학습 가중치를 줍니다.
base_model = RandomForestClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

# [6] Grid는 3×4=12개의 이산 조합을 모두 확인합니다.
grid_space = {
    "n_estimators": [80, 140, 200],
    "max_depth": [4, 6, 8, 10],
}

# [7] Random은 같은 최소·최대 경계의 더 촘촘한 목록에서 12개를 뽑습니다.
random_space = {
    "n_estimators": list(range(80, 201, 10)),
    "max_depth": list(range(4, 11)),
}

print("dev/test rows:", len(y_dev), len(y_test))
print("needs_review rate:", round(float(y_dev.mean()), 3))
print("sealed test used for scoring:", False)

assert len(y_dev) == 1280
assert len(y_test) == 320
assert X_dev.shape[1] == 14

dev/test rows: 1280 320
needs_review rate: 0.123
sealed test used for scoring: False


In [2]:
def build_and_fit_searches(
    base_model,
    grid_space,
    random_space,
    cv,
    X_dev,
    y_dev,
):
    """같은 비교 조건으로 Grid와 Random을 학습하고 예산표를 반환합니다."""
    # Grid는 12개의 이산 조합을 빠짐없이 평가합니다.
    grid = GridSearchCV(
        estimator=base_model,
        param_grid=grid_space,
        scoring="average_precision",
        cv=cv,
        n_jobs=1,
        refit=False,
    )

    # Random은 더 촘촘한 후보 목록에서 12개 조합을 재현 가능하게 뽑습니다.
    random = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=random_space,
        n_iter=12,
        scoring="average_precision",
        cv=cv,
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=False,
    )

    searches = {"Grid": grid, "Random": random}

    # 후보 선택에는 개발 데이터만 사용합니다.
    # X_test와 y_test는 이 함수의 인자에도 포함하지 않았습니다.
    for search in searches.values():
        search.fit(X_dev, y_dev)

    # 실제 평가가 끝난 후보 수를 cv_results_에서 읽습니다.
    folds = cv.get_n_splits()
    budget_rows = []
    for method, search in searches.items():
        candidate_count = len(search.cv_results_["params"])
        budget_rows.append({
            "method": method,
            "candidates": candidate_count,
            "folds": folds,
            "CV_fits": candidate_count * folds,
        })

    budget = pd.DataFrame(budget_rows)

    # 점수와 별개로 비교 예산 계약을 자동 검증합니다.
    assert budget["candidates"].tolist() == [12, 12]
    assert budget["CV_fits"].tolist() == [48, 48]
    assert int(budget["CV_fits"].sum()) == 96

    return searches, budget


searches, budget = build_and_fit_searches(
    base_model,
    grid_space,
    random_space,
    cv,
    X_dev,
    y_dev,
)

print(budget.to_string(index=False))
print("total CV fits:", int(budget["CV_fits"].sum()))

method  candidates  folds  CV_fits
  Grid          12      4       48
Random          12      4       48
total CV fits: 96


In [3]:
def summarize_best_candidates(searches: dict) -> pd.DataFrame:
    """탐색 방식별 최고 후보의 평균·변동·파라미터를 요약합니다."""
    rows = []

    for method, search in searches.items():
        # best_index_는 해당 탐색 안에서 평균 AP가 가장 높은 후보 행입니다.
        best_index = search.best_index_
        results = search.cv_results_
        params = results["params"][best_index]

        rows.append({
            "method": method,
            "best_mean_AP": float(results["mean_test_score"][best_index]),
            "best_std_AP": float(results["std_test_score"][best_index]),
            "n_estimators": int(params["n_estimators"]),
            "max_depth": int(params["max_depth"]),
        })

    return pd.DataFrame(rows)


def select_and_evaluate_once(
    summary,
    X_dev,
    y_dev,
    X_test,
    y_test,
):
    """CV 결과로 후보를 고정한 뒤 test AP를 한 번 계산합니다."""
    # 전체 최고 평균을 찾고, 그 값에서 0.01 이내인 후보만 남깁니다.
    best_mean = float(summary["best_mean_AP"].max())
    eligible = summary[
        summary["best_mean_AP"] >= best_mean - MEAN_TOLERANCE
    ].copy()

    # 평균 허용 범위 안에서는 fold 변동 → 트리 수 → 깊이 → 이름 순으로 고릅니다.
    selected = eligible.sort_values(
        SELECTION_ORDER,
        kind="mergesort",
    ).iloc[0]

    selected_params = {
        "n_estimators": int(selected["n_estimators"]),
        "max_depth": int(selected["max_depth"]),
    }

    # 후보 비교를 끝낸 뒤 선택된 설정 하나만 개발 데이터 전체로 재학습합니다.
    final_model = RandomForestClassifier(
        **selected_params,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    final_model.fit(X_dev, y_dev)

    # 여기서 처음으로 봉인 test의 성능을 계산합니다.
    positive_index = final_model.classes_.tolist().index(1)
    test_score = final_model.predict_proba(X_test)[:, positive_index]
    test_ap = average_precision_score(y_test, test_score)

    # 선택 후보가 사전 허용 범위 안에 있는지 자동 확인합니다.
    assert float(selected["best_mean_AP"]) >= best_mean - MEAN_TOLERANCE
    assert 0.0 <= test_ap <= 1.0

    return selected, len(eligible), test_ap


summary = summarize_best_candidates(searches)
selected, eligible_count, test_ap = select_and_evaluate_once(
    summary,
    X_dev,
    y_dev,
    X_test,
    y_test,
)

print(summary.to_string(
    index=False,
    formatters={
        "best_mean_AP": "{:.3f}".format,
        "best_std_AP": "{:.3f}".format,
    },
))
print("eligible candidates:", eligible_count)
print("selected method:", selected["method"])
print("selected params:", {
    "n_estimators": int(selected["n_estimators"]),
    "max_depth": int(selected["max_depth"]),
})
print("sealed test AP:", f"{test_ap:.3f}")

method best_mean_AP best_std_AP  n_estimators  max_depth
  Grid        0.618       0.042           200         10
Random        0.616       0.042           180         10
eligible candidates: 2
selected method: Random
selected params: {'n_estimators': 180, 'max_depth': 10}
sealed test AP: 0.731
